# Extension results — figures and tables

Three experiments, one section each:

1. **Post-LLM bias** — how positively C4-style web text talks about ethnic/religious groups in 2019 vs Feb 2024 vs Dec 2024.
2. **Gender & age** — who the 2019-C4-trained model (UnifiedQA-3B) puts in a positive light: women or men, which ages, which ethnicities.
3. **Machine text** — how much of the web looks machine-written in 2019 vs Feb 2024 vs Dec 2024 (Binoculars detector + "LLM marker words").

**Run this from the `extension/` folder** (local Jupyter / local Colab runtime). All paths are relative to it.
In hosted Colab: mount Drive, `%cd` into the `extension` folder, then run all.

Everything is written to `extension/results/` — PNG at 300 dpi, tables as CSV + PNG.

In [ ]:
%pip -q install wordcloud

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# --- paths (relative to extension/) ---
EXT = Path.cwd()
if not (EXT / 'post_llm_bias').is_dir() and (EXT / 'extension').is_dir():
    EXT = EXT / 'extension'          # launched from the repo root
ZIPS = {
    'bias': EXT / 'post_llm_bias' / 'post_llm_bias_results.zip',
    'ga1':  EXT / 'gender_age' / 'ga_probe_account1.zip',
    'ga2':  EXT / 'gender_age' / 'ga_probe_account2.zip',
    'mt':   EXT / 'machine_text' / 'machine_text_results.zip',
}
for k, p in ZIPS.items():
    assert p.exists(), f'missing {p.relative_to(EXT)} - run this notebook from the extension/ folder'
OUT = EXT / 'results'
OUT.mkdir(exist_ok=True)
print('extension folder:', EXT)
print('writing to      :', OUT)

def read_csv(zip_key, member):
    with zipfile.ZipFile(ZIPS[zip_key]) as z, z.open(member) as f:
        return pd.read_csv(f)

# --- style: minimal, consistent ---
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '-',
    'legend.frameon': False,
})
LABEL = {'2019': '2019', '2019 C4': '2019',
         'CC-MAIN-2024-10': 'Feb 2024', 'CC-MAIN-2024-51': 'Dec 2024',
         '2024 pooled': '2024 (Feb+Dec)'}
TIME = ['2019', 'Feb 2024', 'Dec 2024']
TCOL = {'2019': '#8c9bab', 'Feb 2024': '#f2a541', 'Dec 2024': '#c8553d', '2024 (Feb+Dec)': '#c8553d'}
GCOL = {'female': '#8e5ea2', 'male': '#3e95cd'}
UP, DOWN, NEUTRAL, HL = '#2a9d8f', '#e76f51', '#bbbbbb', '#fff3b0'

# --- helpers ---
def save_fig(fig, section, name):
    d = OUT / section
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f'{name}.png', dpi=300)
    plt.show()
    plt.close(fig)

def maybe_log(ax, values, axis='y', ratio=20):
    # switch to log scale only when positive values span more than `ratio`x
    v = np.asarray(values, float)
    v = v[np.isfinite(v) & (v > 0)]
    if len(v) > 1 and v.max() / v.min() > ratio:
        (ax.set_yscale if axis == 'y' else ax.set_xscale)('log')
        return True
    return False

def save_table(df, section, name, title, row_colors=None, hl_cols=()):
    # CSV + 300-dpi PNG; row_colors = list of colours/None per row, hl_cols = columns to shade
    d = OUT / section
    d.mkdir(parents=True, exist_ok=True)
    df = df.reset_index(drop=True)
    df.to_csv(d / f'{name}.csv', index=False)
    shown = df.copy()
    for c in shown.columns:
        if pd.api.types.is_float_dtype(shown[c]):
            shown[c] = shown[c].map(lambda x: '' if pd.isna(x) else f'{x:.1f}')
    fig, ax = plt.subplots(figsize=(max(6, 1.25 * len(df.columns)), 0.32 * (len(df) + 1) + 0.7))
    ax.axis('off')
    tbl = ax.table(cellText=shown.astype(str).values, colLabels=list(df.columns),
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.35)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#dddddd')
        if r == 0:
            cell.set_facecolor('#eeeeee'); cell.set_text_props(weight='bold')
            continue
        if df.columns[c] in hl_cols:
            cell.set_facecolor(HL)
        if row_colors is not None and row_colors[r - 1]:
            cell.set_facecolor(row_colors[r - 1])
    ax.set_title(title, fontweight='bold', pad=6)
    fig.savefig(d / f'{name}.png', dpi=300)
    plt.close(fig)
    sty = shown.style.hide(axis='index').set_caption(title)
    if row_colors is not None:
        sty = sty.apply(lambda s: [f'background-color:{row_colors[s.name]}' if row_colors[s.name] else '' for _ in s], axis=1)
    for c in hl_cols:
        sty = sty.set_properties(subset=[c], **{'background-color': HL})
    display(sty)

---
## 1. Post-LLM bias: has the web's tone towards groups changed since 2019?

**Data:** 12 shards of 2019 C4 vs 24 + 24 shards of C4 rebuilt from Common Crawl Feb 2024 (`CC-MAIN-2024-10`) and Dec 2024 (`CC-MAIN-2024-51`).
**Score:** % of a group's paragraphs that read positive, **minus** the same % for random paragraphs (baseline).
Below zero = the group is written about less positively than an average paragraph → *marginalized*.

In [ ]:
pos  = read_csv('bias', 'analysis/positivity.csv')
chg  = read_csv('bias', 'analysis/change.csv')
gap  = read_csv('bias', 'analysis/gap.csv')
disp = read_csv('bias', 'analysis/dispersion.csv')
for df in (pos, chg, gap, disp):
    df['when'] = df['corpus'].map(LABEL)

# groups with enough paragraphs in every corpus (small groups give unreadable CIs)
ok = pos[pos.target != 'Baseline'].groupby('target').enough_paragraphs_everywhere.all()
GROUPS = [g for g in ok[ok].index if g != 'Arab (non-geopolitical)']
print('groups used:', GROUPS)
print('dropped (too few paragraphs):', list(ok[~ok].index))

### 1.1 Which measure to use

Four sentiment measures were run. We use the one that **separates groups the most in 2019** (largest spread between groups after removing sampling noise) — it is the most sensitive to bias.
Set `BEST = '...'` below to override.

In [ ]:
d = disp.pivot(index='measure', columns='when', values='sd_noise_corrected')
r = disp[disp.corpus == '2024 pooled'].set_index('measure').spearman_vs_ref
BEST = d['2019'].idxmax()
# BEST = 'RoBERTa'   # override here

t = pd.DataFrame({
    'measure': d.index,
    'spread between groups, 2019 (pp)': d['2019'].values,
    'spread, 2024 (pp)': d['2024 (Feb+Dec)'].values,
    'same group ranking 2019 vs 2024 (Spearman)': r.reindex(d.index).values,
}).sort_values('spread between groups, 2019 (pp)', ascending=False)
save_table(t, 'post_llm_bias', 'T1_1_measure_choice',
           f'Measure choice ({BEST})',
           row_colors=[HL if m == BEST else None for m in t.measure])
print('BEST measure:', BEST)

### 1.2 % of paragraphs that read positive, per group
One panel per crawl. Each bar = % of that group's paragraphs that read positive.
Dashed line = the same % for random paragraphs (the "average paragraph"). Red bar = below average (marginalized), green = above.

In [ ]:
p = pos[(pos.measure == BEST) & pos.target.isin(GROUPS) & pos.when.isin(TIME)]
w = p.pivot(index='target', columns='when', values='relative')[TIME].sort_values('2019')   # order: most marginalized in 2019 first
raw = p.pivot(index='target', columns='when', values='positivity')[TIME].loc[w.index]
base = pos[(pos.measure == BEST) & (pos.target == 'Baseline')].set_index('when').positivity

fig, axes = plt.subplots(1, 3, figsize=(12, 0.38 * len(raw) + 1.4), sharey=True)
for ax, t_ in zip(axes, TIME):
    ax.barh(raw.index, raw[t_], color=np.where(raw[t_] < base[t_], DOWN, UP))
    ax.axvline(base[t_], ls='--', color='k', lw=0.9)
    ax.text(base[t_], len(raw) - 0.4, f' average {base[t_]:.0f}%', fontsize=8, va='bottom')
    ax.set_xlim(0, 100)
    ax.set_title(t_)
    ax.set_xlabel('% positive')
    ax.grid(axis='y', visible=False)
fig.suptitle(f'Positive paragraphs per group ({BEST})', fontweight='bold')
save_fig(fig, 'post_llm_bias', 'F1_2_positive_share_per_group')

g0 = w.index[0]
print(f"Reading example: in 2019, {raw.loc[g0, '2019']:.0f}% of {g0} paragraphs read positive vs "
      f"{base['2019']:.0f}% of average paragraphs -> {g0} is {-w.loc[g0, '2019']:.0f} points below average.")

### 1.3 Most marginalized groups over time
The four groups furthest below the average paragraph in 2019. y = how many percentage points below the average paragraph they sit (0 = average).

In [ ]:
low4 = w.index[:4]
fig, ax = plt.subplots(figsize=(6.5, 4))
for g, c in zip(low4, ['#c8553d', '#f2a541', '#2a9d8f', '#577590']):
    q = w.loc[g, TIME]
    ax.plot(TIME, q.values, marker='o', color=c, lw=2)
    for xi, v in enumerate(q.values):
        ax.annotate(f'{v:.0f}', (xi, v), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8, color=c)
    ax.annotate(g, (2, q.values[-1]), textcoords='offset points', xytext=(10, 0), va='center', color=c, fontweight='bold')
ax.axhline(0, ls='--', color='k', lw=0.8)
ax.text(0, 0.5, 'average paragraph', fontsize=8, va='bottom')
ax.set_xlim(-0.2, 2.6)
ax.set_ylabel('points vs. average paragraph')
ax.set_title(f'Most marginalized groups ({BEST})')
save_fig(fig, 'post_llm_bias', 'F1_3_marginalized_trend')

### 1.4 How much each group's tone shifted, 2019 → 2024
Change in positivity **beyond the general shift of the whole web** (2024 Feb+Dec vs 2019).
Coloured = the change is statistically clear (95% CI excludes 0): green = more positive now, red = more negative.

In [ ]:
c = chg[(chg.measure == BEST) & (chg.corpus == '2024 pooled') & chg.target.isin(GROUPS)].sort_values('change_vs_baseline')
col = [UP if lo > 0 else DOWN if hi < 0 else NEUTRAL for lo, hi in zip(c.dd_lo, c.dd_hi)]
fig, ax = plt.subplots(figsize=(7, 0.4 * len(c) + 1.2))
ax.barh(c.target, c.change_vs_baseline, color=col)
ax.errorbar(c.change_vs_baseline, c.target, xerr=[c.change_vs_baseline - c.dd_lo, c.dd_hi - c.change_vs_baseline],
            fmt='none', ecolor='#555555', lw=0.8, capsize=0)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('shift in positivity, 2019 -> 2024 (pp, relative to the whole web)')
ax.set_title(f'Shift per group ({BEST})')
ax.grid(axis='y', visible=False)
save_fig(fig, 'post_llm_bias', 'F1_4_shift_per_group')

### 1.5 Jewish vs Arab paragraphs
For each crawl: % of Jewish paragraphs that read positive, % of Arab paragraphs, and the average paragraph for reference.
The number above each group of bars is the gap (Jewish − Arab, in points).
Right panel keeps only Arab paragraphs that are **not** about conflict/politics, to separate news events from everyday framing.

In [ ]:
COLS = {'Jewish': '#577590', 'Arab': '#c8553d', 'average paragraph': '#cccccc'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, arab, ttl in [(axes[0], 'Arab', 'all Arab paragraphs'),
                      (axes[1], 'Arab (non-geopolitical)', 'Arab paragraphs, non-geopolitical')]:
    v = pos[(pos.measure == BEST) & pos.when.isin(TIME)].pivot(index='when', columns='target', values='positivity').loc[TIME]
    v = v[['Jewish', arab, 'Baseline']].set_axis(list(COLS), axis=1)
    x = np.arange(len(TIME)); bw = 0.27
    for i, (k, c_) in enumerate(COLS.items()):
        ax.bar(x + (i - 1) * bw, v[k], bw, color=c_, label=k)
    gq = gap[(gap.arab_paragraphs == arab) & (gap.measure == BEST)].set_index('when').gap
    for xi, t_ in enumerate(TIME):
        ax.text(xi - bw / 2, v.loc[t_, ['Jewish', 'Arab']].max() + 2, f'gap {gq[t_]:+.0f}', ha='center', fontsize=9)
    ax.set_xticks(x, TIME)
    ax.set_ylim(0, 100)
    ax.set_title(ttl)
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('% of paragraphs that read positive')
axes[0].legend(loc='lower left', ncol=3, fontsize=8)
fig.suptitle(f'Jewish vs Arab ({BEST})', fontweight='bold')
save_fig(fig, 'post_llm_bias', 'F1_5_jewish_arab')

### 1.6 Key-result tables

In [ ]:
# T1.2 marginalization ranking (1 = most marginalized)
rk = pos[(pos.measure == BEST) & pos.target.isin(GROUPS)].pivot(index='target', columns='when', values='relative')
t = pd.DataFrame({
    'group': rk.index,
    '2019 (pp)': rk['2019'].values, 'Feb 2024 (pp)': rk['Feb 2024'].values, 'Dec 2024 (pp)': rk['Dec 2024'].values,
    'rank 2019': rk['2019'].rank().astype(int).values,
    'rank 2024 (Feb+Dec)': rk['2024 (Feb+Dec)'].rank().astype(int).values,
}).sort_values('rank 2019')
save_table(t, 'post_llm_bias', 'T1_2_marginalization_ranking',
           f'Marginalization ranking ({BEST})',
           row_colors=['#f8d7cf' if r_ <= 3 else None for r_ in t['rank 2019']])

# T1.3 every statistically clear shift (any measure) - BEST rows highlighted
s = chg[chg.target.isin(GROUPS + ['Arab (non-geopolitical)']) & ((chg.dd_lo > 0) | (chg.dd_hi < 0))]
s = s.assign(direction=np.where(s.change_vs_baseline > 0, 'more positive', 'more negative'))
t = s[['when', 'target', 'measure', 'change_vs_baseline', 'dd_lo', 'dd_hi', 'direction']].rename(columns={
    'when': 'crawl', 'target': 'group', 'change_vs_baseline': 'shift vs 2019 (pp)', 'dd_lo': 'CI low', 'dd_hi': 'CI high'})
t = t.sort_values(['group', 'crawl', 'measure'])
save_table(t, 'post_llm_bias', 'T1_3_significant_shifts',
           'Significant shifts since 2019',
           row_colors=[HL if m == BEST else None for m in t.measure])

# T1.4 Jewish-Arab gap
g = gap[gap.measure == BEST][['arab_paragraphs', 'when', 'gap', 'ci_lo', 'ci_hi', 'change_vs_ref']]
g.columns = ['Arab paragraphs', 'crawl', 'gap (pp)', 'CI low', 'CI high', 'change vs 2019 (pp)']
save_table(g, 'post_llm_bias', 'T1_4_jewish_arab_gap', f'Jewish-Arab gap ({BEST})', hl_cols=['gap (pp)'])

---
## 2. Gender & age: who does the 2019-C4 model put in a positive light?

**Probe:** UnifiedQA-3B (trained on 2019 C4) answers questions like *"A woman and a man … who was honest?"*.
**Score:** % of questions where a subject is picked for the **positive** attribute. Each question pits two subjects, so **50% = no preference**;
above 50 = favoured, below 50 = put in a negative light.

In [ ]:
ga_all = read_csv('ga2', 'results/P1_unifiedqa3b/analysis/gender_age/contrasts.csv')
ge_all = read_csv('ga2', 'results/P1_unifiedqa3b/analysis/gender_ethnicity/contrasts.csv')
age_c  = read_csv('ga2', 'results/P1_unifiedqa3b/analysis/age/contrasts.csv')
ga_sub = read_csv('ga2', 'results/P1_unifiedqa3b/analysis/gender_age/subjects.csv')
eth_f  = read_csv('ga2', 'results/P1_unifiedqa3b/analysis/gender_ethnicity/female_positivity_by_ethnicity.csv')
names  = read_csv('ga1', 'results/P1_unifiedqa3b/analysis/gender_names/contrasts.csv')
nouns  = read_csv('ga1', 'results/P1_unifiedqa3b/analysis/gender_nouns/contrasts.csv')
noun_s = read_csv('ga1', 'results/P1_unifiedqa3b/analysis/gender_nouns/subjects.csv')
AGE_ORDER = ['teen', 'young_adult', 'middle_aged', 'elderly']
AGE_LBL = {'teen': 'teen', 'young_adult': 'young adult', 'middle_aged': 'middle-aged', 'elderly': 'elderly'}
SHOW_CI = True   # thin CI whiskers on bars; set False for the cleanest look
ALL = 'all attributes'

### 2.1 Women vs men, by ethnicity
Within every ethnicity, how often the woman vs the man was picked for the positive attribute. ★ = this ethnicity's women differ clearly from the average woman.

In [ ]:
e = ge_all[(ge_all.axis == 'gender') & (ge_all.subset == ALL) & ge_all.within.str.startswith('eth=')].copy()
e['eth'] = e.within.str[4:]
wide = e.pivot(index='eth', columns='value', values='positivity').sort_values('female', ascending=False)
lo = e.pivot(index='eth', columns='value', values='ci_lo').loc[wide.index]
hi = e.pivot(index='eth', columns='value', values='ci_hi').loc[wide.index]

x = np.arange(len(wide)); bw = 0.4
fig, ax = plt.subplots(figsize=(11, 4.2))
for i, gnd in enumerate(['female', 'male']):
    xx = x + (i - 0.5) * bw
    ax.bar(xx, wide[gnd], bw, color=GCOL[gnd], label='woman' if gnd == 'female' else 'man')
    if SHOW_CI:
        ax.errorbar(xx, wide[gnd], yerr=[wide[gnd] - lo[gnd], hi[gnd] - wide[gnd]], fmt='none', ecolor='#555555', lw=0.7)
for xi, eth in zip(x, wide.index):
    if eth_f.set_index('ethnicity').anomaly.get(eth, False):
        ax.text(xi - bw / 2, hi.loc[eth, 'female'] + 1.5, '★', ha='center', color='k')
ax.axhline(50, ls='--', color='k', lw=0.8)
ax.set_xticks(x, wide.index, rotation=35, ha='right')
ax.set_ylabel('% picked for the positive attribute')
ax.set_ylim(0, 100)
ax.set_title('Women vs men, by ethnicity')
ax.legend(ncol=2, loc='upper right')
ax.grid(axis='x', visible=False)
save_fig(fig, 'gender_age', 'F2_1_gender_by_ethnicity')

### 2.2 Age
How often each age group is picked for the positive attribute (left), and split by gender (right).

In [ ]:
a = age_c[(age_c.subset == ALL) & (age_c.within == 'all')].set_index('value').loc[AGE_ORDER]
ag = ga_all[(ga_all.axis == 'age') & (ga_all.subset == ALL) & ga_all.within.str.startswith('gender=')].copy()
ag['gender'] = ag.within.str[7:]
agw = ag.pivot(index='value', columns='gender', values='positivity').loc[AGE_ORDER]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax1.bar([AGE_LBL[v] for v in a.index], a.positivity, color='#577590')
if SHOW_CI:
    ax1.errorbar(range(len(a)), a.positivity, yerr=[a.positivity - a.ci_lo, a.ci_hi - a.positivity], fmt='none', ecolor='#555555', lw=0.7)
ax1.set_title('By age')
ax1.set_ylabel('% picked for the positive attribute')
x = np.arange(len(agw)); bw = 0.4
for i, gnd in enumerate(['female', 'male']):
    ax2.bar(x + (i - 0.5) * bw, agw[gnd], bw, color=GCOL[gnd], label='women' if gnd == 'female' else 'men')
ax2.set_xticks(x, [AGE_LBL[v] for v in agw.index])
ax2.set_title('By age and gender')
ax2.legend()
for ax in (ax1, ax2):
    ax.axhline(50, ls='--', color='k', lw=0.8); ax.set_ylim(0, 100); ax.grid(axis='x', visible=False)
fig.suptitle('Age', fontweight='bold')
save_fig(fig, 'gender_age', 'F2_2_age')

### 2.3 Every gender × age subject, ranked

In [ ]:
s = ga_sub.sort_values('positivity')
fem = s.subject.str.contains('woman|girl')
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.barh(s.subject, s.positivity, color=np.where(fem, GCOL['female'], GCOL['male']))
ax.axvline(50, ls='--', color='k', lw=0.8)
ax.set_xlim(0, 100)
ax.set_xlabel('% picked for the positive attribute')
ax.set_title('Gender x age subjects')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GCOL['female']), plt.Rectangle((0, 0), 1, 1, color=GCOL['male'])],
          labels=['female', 'male'], loc='lower right')
ax.grid(axis='y', visible=False)
save_fig(fig, 'gender_age', 'F2_3_subjects_ranked')

### 2.4 Women's score across probes
Women's score when the subject is written as a first name, a noun (*mother/father*), with an age, or with an ethnicity.

In [ ]:
probes = {'first names': names, 'nouns (mother, son...)': nouns,
          'with age': ga_all[ga_all.axis == 'gender'], 'with ethnicity': ge_all[ge_all.axis == 'gender']}
rows = []
for k, df in probes.items():
    r_ = df[(df.subset == ALL) & (df.within == 'all') & (df.value == 'female')].iloc[0]
    rows.append((k, r_.positivity, r_.ci_lo, r_.ci_hi))
pr = pd.DataFrame(rows, columns=['probe', 'female', 'lo', 'hi'])
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.barh(pr.probe, pr.female, color=GCOL['female'])
if SHOW_CI:
    ax.errorbar(pr.female, pr.probe, xerr=[pr.female - pr.lo, pr.hi - pr.female], fmt='none', ecolor='#555555', lw=0.7)
ax.axvline(50, ls='--', color='k', lw=0.8)
ax.set_xlim(0, 100)
ax.set_xlabel("women's % picked for the positive attribute")
ax.set_title('Women, by probe')
ax.grid(axis='y', visible=False)
save_fig(fig, 'gender_age', 'F2_4_gender_all_probes')

### 2.5 Key-result tables

In [ ]:
# T2.1 most favoured vs most marginalized on every axis
def extremes(df, label_col='value', score='positivity'):
    d_ = df.sort_values(score)
    return d_.iloc[-1][label_col], d_.iloc[-1][score], d_.iloc[0][label_col], d_.iloc[0][score]

e_m = e[e.value == 'male'].assign(value=lambda d_: d_.eth)
e_f = e[e.value == 'female'].assign(value=lambda d_: d_.eth)
rows = [
    ('Gender (first names)', *extremes(names[(names.subset == ALL) & (names.within == 'all')])),
    ('Gender (nouns)', *extremes(nouns[(nouns.subset == ALL) & (nouns.within == 'all')])),
    ('Gender noun', *extremes(noun_s, 'subject')),
    ('Age', *extremes(age_c[(age_c.subset == ALL) & (age_c.within == 'all')])),
    ('Gender x age subject', *extremes(ga_sub, 'subject')),
    ('Ethnicity, among women', *extremes(e_f)),
    ('Ethnicity, among men', *extremes(e_m)),
]
t = pd.DataFrame(rows, columns=['axis', 'most favoured', 'score', 'most marginalized', 'score (marg.)'])
t['most favoured'] = t['most favoured'].replace(AGE_LBL)
t['most marginalized'] = t['most marginalized'].replace(AGE_LBL)
save_table(t, 'gender_age', 'T2_1_favoured_vs_marginalized',
           'Most favoured vs most marginalized',
           hl_cols=['most marginalized', 'score (marg.)'])

# T2.2 ethnicities whose women differ clearly from the average woman
t = eth_f[['ethnicity', 'positivity', 'minus_pooled', 'diff_lo', 'diff_hi', 'p_holm', 'anomaly']].rename(columns={
    'positivity': "women's score", 'minus_pooled': 'vs average woman (pp)', 'diff_lo': 'CI low',
    'diff_hi': 'CI high', 'p_holm': 'p (Holm)', 'anomaly': 'clear difference'})
t['p (Holm)'] = t['p (Holm)'].map(lambda v: f'{v:.3f}')
save_table(t, 'gender_age', 'T2_2_ethnicity_anomalies',
           "Women by ethnicity vs average woman",
           row_colors=[(UP + '55' if d_ > 0 else DOWN + '55') if a_ else None
                       for d_, a_ in zip(t['vs average woman (pp)'], t['clear difference'])])

---
## 3. Machine text: how much of the web looks machine-written?

**Data:** 2019 C4 vs Feb 2024 vs Dec 2024 (4 pilot shards per 2024 crawl — confidence intervals are wide).
Two independent signals:
- **Binoculars** — an AI-text detector. Its threshold is set so that only 1% of 2019 C4 is flagged; anything **above** that in 2024 is extra machine-like text.
- **Marker words** — words LLMs overuse (*delve, tapestry, showcase, meticulous…*). Ordinary **control** words (*however, people…*) should stay flat.

In [ ]:
mr   = read_csv('mt', 'analysis/marker_rates.csv')
bf   = read_csv('mt', 'analysis/binoculars_flagged.csv')
bg   = read_csv('mt', 'analysis/binoculars_by_group.csv')
lb   = read_csv('mt', 'analysis/llm_share_lower_bound.csv')
tone = read_csv('mt', 'analysis/tone.csv')
for df in (mr, bf, bg, tone):
    df['when'] = df['corpus'].map(LABEL)

### 3.1 Estimated share of machine-written text
Extra share above 2019, from each signal. 2019 is the reference (0 by construction).

In [ ]:
b1 = bf[bf.fpr_on_2019_pct == 1].set_index('when').loc[TIME]
nominal = b1.flagged_pct.loc['2019']
lbm = lb[(lb.compared_with == '2019 C4') & (lb.words == 'ANY MARKER')].assign(when=lambda d_: d_.corpus.map(LABEL)).set_index('when')
est = pd.DataFrame({
    'Binoculars (above 2019 false-alarm rate)': [0, *(b1.flagged_pct.loc[TIME[1:]] - nominal)],
    'marker words (lower bound)': [0, *lbm.loc[TIME[1:], 'lower_bound_llm_share_pct']],
}, index=TIME)
err = pd.DataFrame({
    'Binoculars (above 2019 false-alarm rate)': [[0, 0], *[[b1.flagged_pct[t_] - b1.ci_lo[t_], b1.ci_hi[t_] - b1.flagged_pct[t_]] for t_ in TIME[1:]]],
    'marker words (lower bound)': [[0, 0], *[[lbm.lower_bound_llm_share_pct[t_] - lbm.ci_lo[t_], lbm.ci_hi[t_] - lbm.lower_bound_llm_share_pct[t_]] for t_ in TIME[1:]]],
}, index=TIME)

x = np.arange(len(TIME)); bw = 0.38
fig, ax = plt.subplots(figsize=(6.5, 4))
for i, (col, c_) in enumerate(zip(est.columns, ['#577590', '#c8553d'])):
    yerr = np.array(err[col].tolist()).T
    ax.bar(x + (i - 0.5) * bw, est[col], bw, color=c_, label=col, yerr=yerr, error_kw=dict(lw=0.7, ecolor='#555555'))
ax.set_xticks(x, TIME)
ax.set_ylabel('extra machine-like documents (% of all docs)')
ax.set_title('Extra machine-like text vs 2019')
ax.legend(loc='upper left')
ax.grid(axis='x', visible=False)
save_fig(fig, 'machine_text', 'F3_1_machine_share_estimate')

### 3.2 Binoculars vs marker words
Left: % of documents Binoculars flags (threshold = 1% false alarms on 2019).
Middle: % of documents containing **any** LLM marker word. Right: % containing **any** ordinary control word (should stay flat).
Dashed line = the 2019 level in each panel.

In [ ]:
cols = ['v', 'lo', 'hi']
q1 = bf[bf.fpr_on_2019_pct == 1].set_index('when').loc[TIME][['flagged_pct', 'ci_lo', 'ci_hi']].set_axis(cols, axis=1)
anym = mr[mr.word == 'ANY MARKER'].set_index('when').loc[TIME][['doc_rate_pct', 'ci_lo', 'ci_hi']].set_axis(cols, axis=1)
anyc = mr[mr.word == 'ANY CONTROL'].set_index('when').loc[TIME][['doc_rate_pct', 'ci_lo', 'ci_hi']].set_axis(cols, axis=1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, q, ttl in [(axes[0], q1, 'Binoculars (1% threshold)'), (axes[1], anym, 'LLM marker words'),
                   (axes[2], anyc, 'control words')]:
    bars = ax.bar(TIME, q.v, color=[TCOL[t_] for t_ in TIME],
                  yerr=[q.v - q.lo, q.hi - q.v], error_kw=dict(lw=0.7, ecolor='#555555'))
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
    ax.axhline(q.v['2019'], ls='--', color='k', lw=0.8)
    ax.set_title(ttl)
    ax.set_ylim(0, q.hi.max() * 1.2)
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('% of documents')
save_fig(fig, 'machine_text', 'F3_2_binoculars_vs_markers')

### 3.3 Individual marker words (count plot)
Every marker word: how many documents per 1,000 contain it, in each crawl (sorted by Dec 2024).
Per 1,000 rather than raw counts, because the crawls have different numbers of documents. Log scale switches on automatically when counts differ by more than 20x.

In [ ]:
mk = mr[(mr.kind == 'marker') & ~mr.word.str.startswith('ANY')]
mw = mk.pivot(index='word', columns='when', values='doc_rate_pct')[TIME]
per1k = (mw * 10).sort_values('Dec 2024', ascending=False)
x = np.arange(len(per1k)); bw = 0.27
fig, ax = plt.subplots(figsize=(max(10, 0.42 * len(per1k)), 4.2))
for i, t_ in enumerate(TIME):
    ax.bar(x + (i - 1) * bw, per1k[t_], bw, color=TCOL[t_], label=t_)
ax.set_xticks(x, per1k.index, rotation=60, ha='right')
log = maybe_log(ax, per1k.values.ravel(), axis='y')
ax.set_ylabel('documents per 1,000' + (' (log scale)' if log else ''))
ax.set_title('Marker words per crawl')
ax.legend(ncol=3, loc='upper right')
ax.grid(axis='x', visible=False)
save_fig(fig, 'machine_text', 'F3_3_marker_words_countplot')

### 3.4 Word clouds of marker words
Word size = how often it appears. Font sizes share one scale across the three panels, so a bigger word really is more frequent.

In [ ]:
from wordcloud import WordCloud
gmax = mw.max().max()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, t_ in zip(axes, TIME):
    freq = {w_: v for w_, v in mw[t_].items() if v > 0}
    wc = WordCloud(width=700, height=450, background_color='white', prefer_horizontal=1.0,
                   max_font_size=int(20 + 100 * mw[t_].max() / gmax), min_font_size=6,
                   relative_scaling=1.0, color_func=lambda *a, c=TCOL[t_], **k: c, random_state=0)
    ax.imshow(wc.generate_from_frequencies(freq), interpolation='bilinear')
    ax.set_title(t_); ax.axis('off')
fig.suptitle('Marker words', fontweight='bold')
save_fig(fig, 'machine_text', 'F3_4_marker_wordclouds')

### 3.5 Tone of machine-like text
Positivity of machine-like documents **minus** the rest (percentage points), with the measure chosen in §1.1.
Above 0 = machine-like documents read more positively. (Differences are small and can be negative, so a linear scale is kept.)

In [ ]:
td = tone[(tone.unit == 'document') & (tone.measure == BEST)].set_index(['when', 'subset']).positivity
diff = pd.DataFrame({
    'Binoculars flagged - not flagged': [td[(t_, 'Binoculars flagged (1% FPR)')] - td[(t_, 'Binoculars not flagged')] for t_ in TIME],
    'has marker word - none': [td[(t_, 'has marker word')] - td[(t_, 'no marker word')] for t_ in TIME],
}, index=TIME)
x = np.arange(len(TIME)); bw = 0.38
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for i, (col, c_) in enumerate(zip(diff.columns, ['#577590', '#c8553d'])):
    bars = ax.bar(x + (i - 0.5) * bw, diff[col], bw, color=c_, label=col)
    ax.bar_label(bars, fmt='%+.1f', padding=2, fontsize=8)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x, TIME)
ax.set_ylabel('positivity difference (pp)')
ax.set_title(f'Tone difference ({BEST})')
ax.legend(loc='upper right', fontsize=8)
ax.grid(axis='x', visible=False)
save_fig(fig, 'machine_text', 'F3_5_tone_difference')

### 3.6 Key-result tables

In [ ]:
# T3.1 Binoculars score summary
t = bf[['fpr_on_2019_pct', 'when', 'docs_scored', 'flagged_pct', 'ci_lo', 'ci_hi', 'excess_over_2019_pct']].rename(columns={
    'fpr_on_2019_pct': 'false alarms on 2019 (%)', 'when': 'crawl', 'docs_scored': 'docs',
    'flagged_pct': 'flagged (%)', 'ci_lo': 'CI low', 'ci_hi': 'CI high', 'excess_over_2019_pct': 'extra vs 2019 (pp)'})
t['false alarms on 2019 (%)'] = t['false alarms on 2019 (%)'].astype(int).astype(str)
clear = (t['CI low'] > t['false alarms on 2019 (%)'].astype(float)) & (t.crawl != '2019')
save_table(t, 'machine_text', 'T3_1_binoculars',
           'Binoculars flag rates',
           row_colors=[HL if c_ else None for c_ in clear])

# T3.2 lower bound on LLM share from marker words
t = lb.assign(crawl=lb.corpus.map(LABEL), vs=lb.compared_with.map(LABEL))[
    ['crawl', 'vs', 'words', 'rate_then_pct', 'rate_now_pct', 'lower_bound_llm_share_pct', 'ci_lo', 'ci_hi']].rename(columns={
    'rate_then_pct': 'rate before (%)', 'rate_now_pct': 'rate now (%)',
    'lower_bound_llm_share_pct': 'LLM share, lower bound (%)', 'ci_lo': 'CI low', 'ci_hi': 'CI high'})
save_table(t, 'machine_text', 'T3_2_llm_share_lower_bound',
           'LLM share, lower bound',
           row_colors=[HL if w_ == 'ANY MARKER' else None for w_ in t.words])

# T3.3 fastest-growing marker words (+0.02 pp floor so words absent in 2019 do not divide by 0)
gr = mw.assign(growth=(mw['Dec 2024'] + 0.02) / (mw['2019'] + 0.02)).sort_values('growth', ascending=False).head(10)
t = gr.reset_index().rename(columns={'word': 'marker word', '2019': '2019 (%)', 'Feb 2024': 'Feb 2024 (%)',
                                      'Dec 2024': 'Dec 2024 (%)', 'growth': 'Dec 2024 / 2019 (x)'})
save_table(t, 'machine_text', 'T3_3_fastest_growing_markers',
           'Fastest-growing marker words',
           row_colors=[HL if g_ >= 3 else None for g_ in t['Dec 2024 / 2019 (x)']])

# T3.4 groups with machine-like documents (only groups with >= 20 docs in that crawl)
t = bg[(bg.docs >= 20) & (bg.flagged_pct > 0)][['when', 'group', 'docs', 'flagged_pct', 'ci_lo', 'ci_hi']].rename(columns={
    'when': 'crawl', 'flagged_pct': 'flagged (%)', 'ci_lo': 'CI low', 'ci_hi': 'CI high'}).sort_values(['crawl', 'flagged (%)'], ascending=[True, False], key=lambda c_: c_.map({t_: i for i, t_ in enumerate(TIME)}) if c_.name == 'crawl' else c_)
save_table(t, 'machine_text', 'T3_4_binoculars_by_group',
           'Binoculars flags by group',
           row_colors=[HL if g_ == 'ANY GROUP' else None for g_ in t.group])

In [ ]:
for p_ in sorted(OUT.rglob('*.png')):
    print(p_.relative_to(EXT))